In [21]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "sanchez2019chimpanzees")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "sanchez2019chimpanzees_updated.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [22]:
import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1)

# df.columns = map(str.lower, df.columns)
# df=df.applymap(lambda s: s.lower() if type(s) == str else s)
# df = df.rename(columns={"specie": "species_original",
#     "dyad": "dyad_original"})
# df['study_id']="sanchez2019chimpanzees"
# df.columns

In [23]:
# df['subject_r'] = df['subject_r'].str.rstrip()
# df['subject_l'] = df['subject_l'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)
    df['participant_2'].replace(x, y, inplace=True)
# df['dyad']=df.subject_r.str.cat(df.subject_l, sep='_')

In [24]:
df['role'].replace("subject_right", "focal_participant_right", inplace=True, regex=True)
df['role_2'].replace("subject_left", "focal_participant_left", inplace=True, regex=True)

# df['role']="focal_participant_right"
# df['role_2']="focal_participant_left"
# df = df.rename(columns={"subject_r": "ape", 
#     "subject_l": "ape_2"})

In [25]:
# comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
# apedf = pd.read_csv(comp_path_ape_info)   
# df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

# comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
# apedf_2 = pd.read_csv(comp_path_ape_info_2)
# df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')
# df.columns

In [26]:
# df = df.rename(columns={"chimp_group": "group", 
#     "efficiency score": "efficiency_score"})
# df.rename(columns={"ape": "participant", "ape_2":"participant_2"}, inplace=True)

# column_transform = [['_l','_left'], ['_r', '_right'], ['_leftat', '_latency']]
# for x,y in column_transform:
#     df.columns = df.columns.str.replace(x, y)
# df.rename(columns={'grapes_left': 'number_grapes_left',  'grapes_right': 'number_grapes_right'}, inplace=True)


In [27]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
df= df.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
df= df.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    df[x] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
    df[x] = pd.to_datetime(df[x])
    df[y] = pd.to_datetime(df[y])
    df[k] = (df[x] - df[y]).dt.days//365


df.rename(columns={"group": "species_subgroup"}, inplace=True)

decimal_list = ['efficiency_score','number_grapes_left','number_grapes_right']
for x in decimal_list:
    df[x]=df[x].astype("Int64")


In [28]:
sanchez2019chimpanzees_standardized=df[['study_id','year', 'month',  'day',
         'participant','age_in_years',
        'sex', 'role', 'participant_2','age_in_years_2', 'sex_2', 'role_2','species','dyad','dyad_sex', 'species_subgroup',
        'phase', 'session', 'trial',  'condition',  
        'pull_left', 'pull_right', 'success_left',
       'success_right', 'efficiency_score', 'number_grapes_left', 'number_grapes_right',
       'latency_pull_left', 'latency_pull_right', 'min_latency', 'coordination']]

In [29]:
comp_out_path_stand = os.path.join(out_pathway, 'sanchez2019chimpanzees_standardized.csv')
sanchez2019chimpanzees_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

In [30]:
names =sanchez2019chimpanzees_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
sanchez2019chimpanzees_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'sanchez2019chimpanzees_glossary.csv')
sanchez2019chimpanzees_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
